# mcPHASES EDA (aggregates only)

**Run this yourself.** Claude is denied this notebook and the data folder (D18, D19, D23).

- Every section prints plain-text tables of aggregates: counts, % missing, percentiles, and
  categories with small groups shown as `<5`. Read them here, and paste them back as they are.
- Never add cells that print rows, `df.head()`, ids, or one participant's values, and never put a
  single participant's series in slides or the writeup.
- Outputs are stripped on commit (nbstripout).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "signals").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from signals.mcphases import data_dir, load  # noqa: E402 (needs ROOT on sys.path)

DATA = data_dir()
print("Reading from:", DATA)
pd.set_option("display.width", 200)

## Helpers

Suppression and summary functions used by every section.

In [ ]:
MIN_CELL = 5  # a category seen in fewer participants than this is shown as "<5"
MAX_LEVELS = 15  # more distinct values = free text or id-like: count only, never the values
VALUE_PCTS = [0.01, 0.25, 0.5, 0.75, 0.99]  # no min/max: an extreme is one person's value
PEOPLE_PCTS = [0.1, 0.25, 0.5, 0.75, 0.9]  # spread across ~42 participants

HORMONES = ["lh", "estrogen", "pdg"]
SELF_REPORT = [
    "flow_volume",
    "flow_color",
    "appetite",
    "exerciselevel",
    "headaches",
    "cramps",
    "sorebreasts",
    "fatigue",
    "sleepissue",
    "moodswing",
    "stress",
    "foodcravings",
    "indigestion",
    "bloating",
]
# file -> (day column, value columns): the signals that line up with the Google Health API
SIGNALS = {
    "wrist_temperature": ("day_in_study", ["temperature_diff_from_baseline"]),
    "computed_temperature": (
        "sleep_end_day_in_study",
        [
            "nightly_temperature",
            "temperature_samples",
            "baseline_relative_sample_sum",
            "baseline_relative_nightly_standard_deviation",
        ],
    ),
    "resting_heart_rate": ("day_in_study", ["value", "error"]),
    "heart_rate_variability_details": (
        "day_in_study",
        ["rmssd", "coverage", "low_frequency", "high_frequency"],
    ),
    "respiratory_rate_summary": (
        "day_in_study",
        ["full_sleep_breathing_rate", "deep_sleep_breathing_rate", "full_sleep_signal_to_noise"],
    ),
    "sleep": ("sleep_end_day_in_study", ["minutesasleep", "timeinbed", "efficiency"]),
    "sleep_score": ("day_in_study", ["overall_score", "resting_heart_rate"]),
}
# One column at a time, binned, never crossed. income and sexually_active are not reported.
DEMOGRAPHICS = ["ethnicity", "self_report_menstrual_health_literacy"]
FILES = ["hormones_and_selfreport", "subject-info", *SIGNALS]


def show(title, df):
    """Print a table as plain text: easy to read here and to paste back."""
    print(f"\n=== {title} ===")
    print(df.to_string() if len(df) else "(empty)")


def pct_missing(values):
    return round(100 * float(values.isna().mean()), 1)


def numeric(values):
    """n, % missing, and 1st to 99th percentiles of one column."""
    nums = pd.to_numeric(values, errors="coerce")
    row = {"n": int(nums.notna().sum()), "% missing": pct_missing(values)}
    for q in VALUE_PCTS:
        row[f"p{round(q * 100)}"] = round(float(nums.quantile(q)), 3) if row["n"] else None
    return row


def across_people(per_person):
    """Spread of a per-participant count across participants (never the per-person list)."""
    row = {"participants": int(per_person.size)}
    for q in PEOPLE_PCTS:
        row[f"p{round(q * 100)}"] = (
            round(float(per_person.quantile(q)), 1) if per_person.size else None
        )
    return row


def categories(df, col):
    """Rows and participants per value. Small groups show as <5; free text is never shown."""
    n_unique = df[col].nunique(dropna=True)
    if n_unique > MAX_LEVELS:
        return pd.DataFrame([{"value": f"({n_unique} distinct values, not shown)"}])
    labels = df[col].astype("object").where(df[col].notna(), "<missing>").astype(str)
    rows = labels.value_counts()
    people = df.groupby(labels)["id"].nunique() if "id" in df else rows
    out = []
    for value, n_rows in rows.items():
        small = people[value] < MIN_CELL
        out.append(
            {
                "value": value,
                "rows": f"<{MIN_CELL} ppl" if small else int(n_rows),
                "participants": f"<{MIN_CELL}" if small else int(people[value]),
            }
        )
    return pd.DataFrame(out)


def suppress(count):
    return count if count >= MIN_CELL else f"<{MIN_CELL}"


def day_range(df, day_col):
    """Where each study interval sits on the day axis: 1st / 50th / 99th percentile."""
    days = pd.to_numeric(df[day_col], errors="coerce")
    return {
        str(k): f"{v.quantile(0.01):.0f} / {v.quantile(0.5):.0f} / {v.quantile(0.99):.0f}"
        for k, v in days.groupby(df["study_interval"])
    }

## 1. Inventory: files, rows, participants, columns, % missing

In [ ]:
summary, columns = [], []
for name in FILES:
    try:
        df = load(name, root=DATA)
    except FileNotFoundError:
        summary.append({"file": name, "rows": "file not found", "participants": "", "columns": ""})
        continue
    by_interval = (
        {str(k): suppress(v) for k, v in df["study_interval"].value_counts().items()}
        if "study_interval" in df
        else ""
    )
    summary.append(
        {
            "file": name,
            "rows": len(df),
            "participants": df["id"].nunique() if "id" in df else None,
            "columns": df.shape[1],
            "rows by interval": by_interval,
        }
    )
    columns += [
        {"file": name, "column": c, "dtype": str(df[c].dtype), "% missing": pct_missing(df[c])}
        for c in df.columns
    ]
show("Files", pd.DataFrame(summary, dtype=object).set_index("file").fillna(""))
show("Columns", pd.DataFrame(columns).set_index(["file", "column"]))

## 2. Phase labels and hormones

In [ ]:
df = load(
    "hormones_and_selfreport", ["id", "day_in_study", "study_interval", "phase", *HORMONES], DATA
)
labeled = df.dropna(subset=["phase"])
tested = df.dropna(subset=[h for h in HORMONES if h in df], how="all")

show("Phase labels", categories(df, "phase"))
hormones = [{"hormone": h, **numeric(df[h])} for h in HORMONES if h in df]
show("Hormones (over all rows)", pd.DataFrame(hormones).set_index("hormone"))
show(
    "Days per participant (spread across participants)",
    pd.DataFrame(
        [
            {"days": "any row", **across_people(df.groupby("id")["day_in_study"].nunique())},
            {
                "days": "with a phase label",
                **across_people(labeled.groupby("id")["day_in_study"].nunique()),
            },
            {
                "days": "with a hormone test",
                **across_people(tested.groupby("id")["day_in_study"].nunique()),
            },
        ]
    ).set_index("days"),
)
by_interval = pd.Series(day_range(df, "day_in_study"), name="days").to_frame()
show("day_in_study by interval (p1 / p50 / p99)", by_interval)

## 3. Self-report (flow, symptoms)

In [ ]:
df = load("hormones_and_selfreport", ["id", *SELF_REPORT], DATA)
table, free_text = {}, []
for col in SELF_REPORT:
    if col in df:
        cats = categories(df, col)
        if "rows" in cats:
            table[col] = {v: str(r) for v, r in zip(cats["value"], cats["rows"], strict=True)}
        else:
            free_text.append(f"{col}: {cats['value'][0]}")
table = pd.DataFrame.from_dict(table, orient="index")
table = table[sorted(table.columns)].fillna("")
show("Rows per response (<5 ppl = fewer than 5 participants gave it)", table)
if free_text:
    print("\nNot shown (free text or too many values):", "; ".join(free_text))

## 4. Wearable signals that line up with the Google Health API

In [ ]:
values, coverage, intervals, absent = [], [], {}, {}
for name, (day_col, value_cols) in SIGNALS.items():
    wanted = ["id", day_col, "study_interval", *value_cols]
    try:
        df = load(name, wanted, DATA)
    except FileNotFoundError:
        absent[name] = "file not found"
        continue
    missing_cols = [c for c in wanted if c not in df]
    if missing_cols:
        absent[name] = ", ".join(missing_cols)
    if name == "computed_temperature" and not missing_cols:
        # Mean nightly deviation from baseline, comparable to the API's nightly minus baseline.
        df["mean_dev_from_baseline"] = (
            df["baseline_relative_sample_sum"] / df["temperature_samples"]
        )
        value_cols = [*value_cols, "mean_dev_from_baseline"]
    values += [{"file": name, "column": c, **numeric(df[c])} for c in value_cols if c in df]
    if day_col in df and "id" in df:
        days = df.dropna(subset=[day_col])
        spread = across_people(days.groupby("id")[day_col].nunique())
        per_day = days.groupby(["id", day_col]).size()
        coverage.append(
            {
                "file": name,
                "participants": spread.pop("participants"),
                **{f"days {k}": v for k, v in spread.items()},
                "rows/day p50": per_day.median(),
                "rows/day p99": per_day.quantile(0.99),
            }
        )
        if "study_interval" in df:
            intervals[name] = day_range(df, day_col)

show("Values", pd.DataFrame(values).set_index(["file", "column"]))
show("Coverage: days with data per participant", pd.DataFrame(coverage).set_index("file"))
show("Day axis by interval (p1 / p50 / p99)", pd.DataFrame(intervals).T)
if absent:
    show("Expected but absent", pd.Series(absent, name="missing").to_frame())

## 5. Who is in the data (one column at a time, binned)

In [ ]:
df = load("subject-info", ["id", "birth_year", "age_of_first_menarche", *DEMOGRAPHICS], DATA)
if "birth_year" in df:
    age = 2022 - pd.to_numeric(df["birth_year"], errors="coerce")  # age at Interval 1
    df["age band (2022)"] = pd.cut(
        age, [0, 20, 25, 30, 35, 200], right=False, labels=["<20", "20-24", "25-29", "30-34", "35+"]
    )
if "age_of_first_menarche" in df:
    menarche = pd.to_numeric(df["age_of_first_menarche"], errors="coerce")
    df["age at first period"] = pd.cut(
        menarche, [0, 11, 13, 15, 100], right=False, labels=["<11", "11-12", "13-14", "15+"]
    )
for col in ["age band (2022)", "age at first period", *DEMOGRAPHICS]:
    if col in df:
        cats = categories(df, col)
        show(f"{col}: participants", cats[[c for c in ("value", "participants") if c in cats]])